In [3]:
from sklearn.linear_model import LogisticRegression
from dataclasses import dataclass
from abc import ABC,abstractmethod      

In [ ]:
# Template

from abc import ABC, abstractmethod
from typing import Union, Dict, Any
import numpy as np

# Mocking scikit-learn imports for demonstration. 
# In a real project, run: pip install scikit-learn
from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score


class BaseModelWrapper(ABC):
    """
    The Abstract Base Class (The Blueprint).
    Every single ML wrapper in your framework MUST implement these methods.
    """
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model: Union[BaseEstimator, None] = None  # Holds the actual scikit-learn object
        self.is_trained: bool = False

    @abstractmethod
    def instantiate_model(self) -> BaseEstimator:
        """Creates and configures the underlying scikit-learn model object."""
        pass

    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Trains the underlying model on the provided training data."""
        pass

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predicts targets for the provided input data."""
        pass

    @abstractmethod
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict[str, float]:
        """Evaluates the model and returns performance metrics."""
        pass

    def get_info(self) -> str:
        """Concrete method: Inherited automatically by all child classes."""
        status = "Trained" if self.is_trained else "Untrained"
        return f"Model Architecture: {self.model_name} | Status: {status}"


class CustomClassifier(BaseModelWrapper):
    """
    A concrete implementation of the Base blueprint for Classification tasks.
    """
    # Class constant to restrict allowed scoring metrics
    VALID_METRICS = {"accuracy", "precision", "recall", "f1"}

    def __init__(self, algorithm_name: str, max_iterations: int, classification_metric: str):
        # 1. Pass core metadata up to the parent Base class
        super().__init__(model_name=algorithm_name)
        
        # 2. Add properties specific to classification
        self.max_iterations = max_iterations
        
        # 3. Input Validation: Catch configuration errors immediately
        if classification_metric.lower() not in self.VALID_METRICS:
            raise ValueError(f"Invalid metric '{classification_metric}'. Choose from: {self.VALID_METRICS}")
        self.classification_metric = classification_metric.lower()
        
        # 4. Trigger the creation of the actual ML model object
        self.model = self.instantiate_model()

    def instantiate_model(self) -> BaseEstimator:
        """Maps string names to actual executable scikit-learn objects."""
        if self.model_name == "LogisticRegression":
            return LogisticRegression(max_iter=self.max_iterations)
        elif self.model_name == "RandomForest":
            return RandomForestClassifier(n_estimators=self.max_iterations) # Using max_iterations as estimators here
        else:
            raise ValueError(f"Unsupported algorithm framework: {self.model_name}")

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        """Trains the actual scikit-learn algorithm."""
        print(f"-> Starting training for {self.model_name}...")
        self.model.fit(X, y)
        self.is_trained = True
        print(f"-> Training complete.")

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Ensures the model is trained before allowing predictions."""
        if not self.is_trained:
            raise RuntimeError("Cannot predict. You must train the model using .fit() first!")
        return self.model.predict(X)

    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray) -> Dict[str, float]:
        """Calculates performance based on your chosen configuration metric."""
        predictions = self.predict(X_test)
        
        # Calculate standard classification metrics
        metrics_dict = {
            "accuracy": float(accuracy_score(y_test, predictions)),
            "precision": float(precision_score(y_test, predictions, average='macro', zero_division=0)),
            "recall": float(recall_score(y_test, predictions, average='macro', zero_division=0)),
            "f1": float(f1_score(y_test, predictions, average='macro', zero_division=0))
        }
        
        # Highlight your primary target metric
        print(f"\n--- Evaluation Results (Primary Metric: {self.classification_metric}) ---")
        return metrics_dict

    def __repr__(self) -> str:
        """Clean object status visualization."""
        return (f"CustomClassifier(algorithm={self.model_name!r}, "
                f"iterations={self.max_iterations}, "
                f"target_metric={self.classification_metric!r}, "
                f"trained={self.is_trained})")


# 1. Create dummy training data (binary classification setup)
X_train = np.array([[1, 2], [2, 3], [3, 4], [5, 1], [6, 2], [7, 3]])
y_train = np.array([0, 0, 0, 1, 1, 1])

X_test = np.array([[1, 1], [6, 4]])
y_test = np.array([0, 1])

# 2. Instantiate your custom wrapper class
lor_pipeline = CustomClassifier(
    algorithm_name="LogisticRegression", 
    max_iterations=500, 
    classification_metric="precision"
)

# 3. View properties and representation
print(lor_pipeline)               # Uses the __repr__ string
print(lor_pipeline.get_info())    # Uses the concrete Base class method

# 4. Execute the training pipeline loop
lor_pipeline.fit(X_train, y_train)

# 5. Evaluate results
results = lor_pipeline.evaluate(X_test, y_test)
print(f"Full Metrics Breakdown: {results}")
print(f"Target Metric Status ({lor_pipeline.classification_metric}): {results[lor_pipeline.classification_metric]}")


CustomClassifier(algorithm='LogisticRegression', iterations=500, target_metric='precision', trained=False)
Model Architecture: LogisticRegression | Status: Untrained
-> Starting training for LogisticRegression...
-> Training complete.

--- Evaluation Results (Primary Metric: precision) ---
Full Metrics Breakdown: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Target Metric Status (precision): 1.0


In [6]:
from abc import ABC, abstractmethod
from typing import Union, Dict, Any
from sklearn.base import BaseEstimator

class Base(ABC):
    "Base from which every other custom model inherits"
    def __init__(self,model_name : str):
        self.model_name = model_name
        self.model: Union[BaseEstimator, None] = None  # Holds the actual scikit-learn object
        self.is_trained: bool = False

    @abstractmethod
    def fit( self,X : np.ndarray , y : np.ndarray ) -> None:
        pass
    @abstractmethod
    def predict( self , X : np.ndarray ):
        pass
    @abstractmethod
    def evaluate_metrics(self,X_t : np.ndarray , y_t : np.ndarray ) -> dict:
        pass
    def get_info(self) -> str:
        """Concrete method: Inherited automatically by all child classes."""
        status = "Trained" if self.is_trained else "Untrained"
        return f"Model Architecture: {self.model_name} | Status: {status}"

class Regressor(Base):
    def __init__(self, model,metric):
        super().__init__(model)

        REG_METRICS = {"r2_score","root_mean_squared_error","mean_absolute_error","mean_squared_error"}
        VALID_MODELS = {"LinearRegression"}

        if metric.lower() not in REG_METRICS :
            raise ValueError(f"Invalid argument : {metric} \nChoose from : {REG_METRICS}")
        self.metric = metric.lower()

        if model.lower() not in VALID_MODELS :
            raise ValueError(f"Invalid argument : {model} \nChoose from : {VALID_MODELS}")
        self.metric = model.lower()        
    

    def fit( self,X : np.ndarray , y : np.ndarray ) -> None:
        self.model.fit(X,y)
        return f"{self.model} fit correctly !!!"

    def predict( self , X : np.ndarray ):
        return self.model.predict(X)

    def evaluate_metrics(self,y_real : np.ndarray , y_predicted : np.ndarray ) -> dict:
        eval_metric = {
            f"{self.metric}" : self.metric(y_real,y_predicted)
        }
        